# LeetCode #1334: Find the City With the Smallest Number of Neighbors at a Threshold Distance

https://leetcode.com/problems/find-the-city-with-the-smallest-number-of-neighbors-at-a-threshold-distance/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force (Dijkstra per city)** | $O(n(n + m) \log n)$ | $O(n + m)$ |
| **Optimal: Floyd-Warshall ★** | $O(n^3)$ | $O(n^2)$ |

---

## Understanding the Methods

### Brute Force (Dijkstra per city)
Run Dijkstra from every city, count reachable cities within `distanceThreshold`. For $n \le 100$ this is acceptable, but Floyd-Warshall is simpler to implement correctly.

### Optimal: Floyd-Warshall ★
Compute all-pairs shortest paths in one $O(n^3)$ pass. For each city, count how many other cities are reachable within the threshold. Return the city with the fewest reachable neighbours (ties broken by highest city index).

**Constraints:**
* $2 \le n \le 100$
* $1 \le edges.length \le n(n-1)/2$
* $1 \le distanceThreshold \le 10^4$

## Solutions

### C#

In [ ]:
public class Solution {
    public int FindTheCity(int n, int[][] edges, int distanceThreshold) {
        // Initialise distance matrix with infinity
        int INF = int.MaxValue / 2;
        var dist = new int[n, n];
        for (int i = 0; i < n; i++)
            for (int j = 0; j < n; j++)
                dist[i, j] = i == j ? 0 : INF;

        // Seed direct edge weights
        foreach (var e in edges) {
            dist[e[0], e[1]] = e[2];
            dist[e[1], e[0]] = e[2];
        }

        // Relax through every intermediate city k
        for (int k = 0; k < n; k++)
            for (int i = 0; i < n; i++)
                for (int j = 0; j < n; j++)
                    if (dist[i, k] + dist[k, j] < dist[i, j])
                        dist[i, j] = dist[i, k] + dist[k, j];

        // Pick the city with fewest reachable neighbours; ties go to highest index
        int result = -1, minReachable = n + 1;
        for (int i = 0; i < n; i++) {
            int reachable = 0;
            for (int j = 0; j < n; j++)
                if (i != j && dist[i, j] <= distanceThreshold) reachable++;
            // ≤ captures ties and naturally keeps the largest index
            if (reachable <= minReachable) { minReachable = reachable; result = i; }
        }
        return result;
    }
}

### Python

In [ ]:
class Solution:
    def findTheCity(self, n: int, edges: list[list[int]], distanceThreshold: int) -> int:
        INF = float('inf')
        # Initialise all-pairs distance matrix
        dist = [[INF] * n for _ in range(n)]
        for i in range(n):
            dist[i][i] = 0
        for u, v, w in edges:
            dist[u][v] = dist[v][u] = w

        # Floyd-Warshall: relax through each intermediate node
        for k in range(n):
            for i in range(n):
                for j in range(n):
                    if dist[i][k] + dist[k][j] < dist[i][j]:
                        dist[i][j] = dist[i][k] + dist[k][j]

        # Find city with minimum reachable neighbours, prefer higher index on tie
        result, min_reach = -1, n + 1
        for i in range(n):
            reachable = sum(1 for j in range(n) if i != j and dist[i][j] <= distanceThreshold)
            if reachable <= min_reach:
                min_reach, result = reachable, i
        return result

### Go

In [ ]:
func findTheCity(n int, edges [][]int, distanceThreshold int) int {
	const INF = 1<<31 - 1
	dist := make([][]int, n)
	for i := range dist {
		dist[i] = make([]int, n)
		for j := range dist[i] {
			if i == j { dist[i][j] = 0 } else { dist[i][j] = INF }
		}
	}
	for _, e := range edges {
		dist[e[0]][e[1]] = e[2]
		dist[e[1]][e[0]] = e[2]
	}

	// Relax paths through each intermediate city
	for k := 0; k < n; k++ {
		for i := 0; i < n; i++ {
			for j := 0; j < n; j++ {
				if dist[i][k] != INF && dist[k][j] != INF {
					if dist[i][k]+dist[k][j] < dist[i][j] {
						dist[i][j] = dist[i][k] + dist[k][j]
					}
				}
			}
		}
	}

	result, minReach := -1, n+1
	for i := 0; i < n; i++ {
		reach := 0
		for j := 0; j < n; j++ {
			if i != j && dist[i][j] <= distanceThreshold { reach++ }
		}
		if reach <= minReach { minReach = reach; result = i }
	}
	return result
}

### Rust

In [ ]:
impl Solution {
    pub fn find_the_city(n: i32, edges: Vec<Vec<i32>>, distance_threshold: i32) -> i32 {
        let n = n as usize;
        const INF: i32 = i32::MAX / 2;
        let mut dist = vec![vec![INF; n]; n];
        for i in 0..n { dist[i][i] = 0; }
        for e in &edges {
            let (u, v, w) = (e[0] as usize, e[1] as usize, e[2]);
            dist[u][v] = w; dist[v][u] = w;
        }

        // Floyd-Warshall all-pairs shortest paths
        for k in 0..n {
            for i in 0..n {
                for j in 0..n {
                    let through_k = dist[i][k].saturating_add(dist[k][j]);
                    if through_k < dist[i][j] { dist[i][j] = through_k; }
                }
            }
        }

        let mut result = -1i32;
        let mut min_reach = n + 1;
        for i in 0..n {
            let reach = (0..n).filter(|&j| j != i && dist[i][j] <= distance_threshold).count();
            if reach <= min_reach { min_reach = reach; result = i as i32; }
        }
        result
    }
}

## Example Scenarios

### 1. Common Case
**Input:** `n=4, edges=[[0,1,3],[1,2,1],[1,3,4],[2,3,1]], distanceThreshold=4`
From city 0: reachable {1,2,3} = 3. From city 3: reachable {1,2} = 2. City 3 wins (fewest, highest index). Answer: **3**.

### 2. Slightly Complex
**Input:** `n=5, edges=[[0,1,2],[0,4,8],[1,2,3],[1,4,2],[2,3,1],[3,4,1]], distanceThreshold=2`
City 0 reaches {1} (dist=2). City 4 reaches {3} (dist=1). Both have 1 neighbour; highest index wins: **4**.

### 3. Edge Case: Time Factor
**Input:** $n=100$, complete graph.
Floyd-Warshall performs $100^3 = 10^6$ operations — comfortably within time limits. Dijkstra-per-city would also work but requires more code.

### 4. Edge Case: Space Factor
**Input:** $n=100$, any graph.
The $100 \times 100$ distance matrix uses $10^4$ integers — $O(n^2)$ space, trivially small.

### 5. Almost-Impossible but Plausible
**Input:** `n=4, edges=[[0,1,10000],[1,2,10000],[2,3,10000]], distanceThreshold=1`
No city can reach any other within threshold 1. Every city has 0 reachable neighbours; the highest-index city (3) is returned: **3**.